# Сохранение фида в postgreSQL

In [1]:
from app.utils.downloader_feed import download_feed
from app.utils.parser_feed import load_products_from_feed
from app.db.save_to_db import save_to_db

download_feed()
df = load_products_from_feed("../feed.xml")
save_to_db(df)

2026-03-12 17:59:53.092 | INFO     | app.downloader_feed:download_feed:19 - Feed file downloaded in 0.63 seconds
2026-03-12 17:59:53.126 | INFO     | app.parser_feed:load_products_from_feed:40 - Dataframe was created in 0.03 seconds
2026-03-12 17:59:53.276 | INFO     | app.save_to_db:save_to_db:18 - 36 products saved to database in 0.15 seconds


,price,categoryId,Тип лампы,Энергосберегающая,Форма колбы,Тип цоколя,Мощность,Напряжение,Свет,Степень пылевлагозащиты,...,Дальность обнаружения,Тип датчика,Минимальное значение диапазона измерения (температура),Максимальное значение диапазона измерения (температура),Диапазон измерения максимального значния влажности,Напряжение питания,Линейка,Тип крепления,Управление,Класс защиты (IP)
0,1490.0,10103,Светодиодная,Да,Свеча,E14,4.8,220,RGB,IP40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1490.0,10103,Светодиодная,Да,Цилиндрическая,E27,NaN,220,RGB,IP40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1490.0,10103,Светодиодная,Да,Софит,GU10,4.9,220,RGB,IP40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8990.0,10101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8990.0,10101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,8990.0,10101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,8990.0,10101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7990.0,10101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,7990.0,10101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,7990.0,10101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Проверка джобы

In [6]:
from app.jobs.download_and_load_df_to_db_job import run_feed_job

run_feed_job()

2026-03-10 21:46:10.440 | INFO     | app.downloader_feed:download_feed:19 - Feed file downloaded in 0.54 seconds
2026-03-10 21:46:10.464 | INFO     | app.parser_fedd:load_products_from_feed:41 - Dataframe was created in 0.02 seconds
2026-03-10 21:46:10.488 | INFO     | app.save_to_db:save_to_db:18 - 36 products saved to database in 0.02 seconds
2026-03-10 21:46:10.489 | INFO     | app.job:run_feed_job:19 - Feed job completed in 0.59 seconds


Запуск по расписанию каждые 10 сек

In [ ]:
import time
from apscheduler.triggers.cron import CronTrigger
from apscheduler.schedulers.background import BackgroundScheduler

from app.jobs.download_and_load_df_to_db_job import run_feed_job

scheduler = BackgroundScheduler()

scheduler.add_job(
    func=run_feed_job,
    trigger=CronTrigger(second="*/10"),
    id="feed_loader_job",
    replace_existing=True,
)

scheduler.start()

while True:
    time.sleep(1)

2026-03-08 13:07:30.317 | INFO     | app.downloader_feed:download_feed:19 - Feed file downloaded in 0.31 seconds
2026-03-08 13:07:30.372 | INFO     | app.parser_fedd:load_products_from_feed:41 - Dataframe was created in 0.04 seconds
2026-03-08 13:07:30.667 | INFO     | app.save_to_db:save_to_db:25 - 36 products saved to database in 0.29 seconds
2026-03-08 13:07:30.669 | INFO     | app.job:run_feed_job:19 - Feed job completed in 0.67 seconds
2026-03-08 13:07:40.216 | INFO     | app.downloader_feed:download_feed:19 - Feed file downloaded in 0.22 seconds
2026-03-08 13:07:40.279 | INFO     | app.parser_fedd:load_products_from_feed:41 - Dataframe was created in 0.06 seconds
2026-03-08 13:07:40.386 | INFO     | app.save_to_db:save_to_db:25 - 36 products saved to database in 0.11 seconds
2026-03-08 13:07:40.388 | INFO     | app.job:run_feed_job:19 - Feed job completed in 0.39 seconds


Прогнозируем цену линейной регрессией

In [4]:
from app.ml.predict_price_by_linear_regression import predict_price_by_linear_regression
from app.db.read_df_from_db import read_df_from_db

df = read_df_from_db()
predict_price_by_linear_regression(df)

2026-03-10 22:55:41.333 | INFO     | app.second.read_df_from_db:read_df_from_db:14 - 72 products saved to dataframe in 0.03 seconds


TypeError: predict_price_by_linear_regression() missing 1 required positional argument: 's3service'

Прогон разных конфигов CatBoost

In [1]:
from app.db.read_df_from_db import read_df_from_db
from app.ml.predict_price_by_catboost import predict_price_by_catboost
from config import *
from minio_service import S3BucketService
from app.ml.split_df_train_and_test import split_df_train_and_test

df = read_df_from_db()

service = S3BucketService(
    MINIO_BUCKET_NAME,
    MINIO_ENDPOINT,
    MINIO_ACCESS_KEY,
    MINIO_SECRET_KEY
)

configs = [
    {"iterations": 200, "depth": 4, "learning_rate": 0.1},
    # {"iterations": 300, "depth": 6, "learning_rate": 0.1},
    # {"iterations": 500, "depth": 6, "learning_rate": 0.05},
    # {"iterations": 700, "depth": 8, "learning_rate": 0.03},
]

x_train, x_test, y_train, y_test = split_df_train_and_test(df)

print(predict_price_by_catboost(x_train, x_test, y_train, y_test , configs, service))


2026-03-14 16:02:47.483 | INFO     | app.db.read_df_from_db:read_df_from_db:14 - 144 products saved to dataframe in 0.1 seconds


[{'iterations': 200, 'depth': 4, 'learning_rate': 0.1, 'metrics': {'RMSE': np.float64(135.51075759890253), 'MAPE': 0.035350671895686504, 'R2': np.float64(0.9998298527777838)}}]


Проверка minio

In [1]:
from minio_service import S3BucketService
from config import *
from app.db.read_df_from_db import read_df_from_db
from app.ml.predict_price_by_linear_regression import predict_price_by_linear_regression
from app.minio.save_datasets import save_datasets
from app.minio.save_linear_regression_model import save_linear_regression_model
from app.ml.split_df_train_and_test import split_df_train_and_test

service = S3BucketService(
    MINIO_BUCKET_NAME,
    MINIO_ENDPOINT,
    MINIO_ACCESS_KEY,
    MINIO_SECRET_KEY
)

df = read_df_from_db()
x_train, x_test, y_train, y_test = split_df_train_and_test(df)
pred_by_LR = predict_price_by_linear_regression(x_train, x_test, y_train, y_test )
print(pred_by_LR[0])

save_linear_regression_model(pred_by_LR[1], service, pred_by_LR[0])
save_datasets(x_train, x_test, pred_by_LR[1], service)


2026-03-14 16:11:32.062 | INFO     | app.db.read_df_from_db:read_df_from_db:14 - 144 products saved to dataframe in 0.1 seconds


{'RMSE': np.float64(1.142010551232025e-11), 'R2': 1.0, 'MAPE': 1.996346495913186e-15}


2026-03-14 16:11:32.553 | INFO     | app.minio.save_datasets:save_datasets:73 - Datasets saved in 0.38 sec


Джоба

In [1]:
from app.jobs.predict_job import run_predict_job

configs = [
    {"iterations": 200, "depth": 4, "learning_rate": 0.1},
    {"iterations": 300, "depth": 6, "learning_rate": 0.1},
    # {"iterations": 500, "depth": 6, "learning_rate": 0.05},
    # {"iterations": 700, "depth": 8, "learning_rate": 0.03},
]
run_predict_job(configs)

2026-03-17 20:38:43.333 | INFO     | app.db.read_df_from_db:read_df_from_db:14 - 108 products saved to dataframe in 0.09 seconds
D:\Second-ML-task\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['Типоразмер' 'Световой поток' 'Принцип работы' 'Количество блоков']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(
D:\Second-ML-task\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['Типоразмер' 'Световой поток' 'Принцип работы' 'Количество блоков']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(
D:\Second-ML-task\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['Типоразмер' 'Световой поток' 'Принцип работы' 'Количество блоков']. At least one non-missing value is needed for imputation with strategy=

{'RMSE': np.float64(433.56564384354795), 'R2': 0.9967218882909008, 'MAPE': 0.15500377574188037}
CATBOOST REGRESSION:  [{'iterations': 200, 'depth': 4, 'learning_rate': 0.1, 'metrics': {'RMSE': np.float64(397.6838037966829), 'MAPE': 0.12609299651418196, 'R2': np.float64(0.9972420281104851)}}, {'iterations': 300, 'depth': 6, 'learning_rate': 0.1, 'metrics': {'RMSE': np.float64(383.1523842035501), 'MAPE': 0.11376916500588626, 'R2': np.float64(0.9974398990440658)}}]
